## 00. Cấu hình chỉnh sửa tập trung

Đổi các đường dẫn Kaggle Input tại đây. `SMOKE_ONLY=True` chỉ chạy kiểm tra 4 mẫu mỗi split và ba bước cập nhật gate. Bật `SMOKE_ONLY=False` cùng các cờ RUN_* để chạy toàn bộ. Mọi stage có kiểm tra kết quả trước khi tiếp tục.


In [1]:
from pathlib import Path
REPO_URL = "https://github.com/vietquang8071/MOESCORE.git"
REPO_REF = "main"  # Hoặc SHA chứa đúng notebook và scripts của MOE-09.5
REPOSITORY_SOURCE = None  # Kaggle Input có source MOESCORE, nếu chưa push mã mới
PROJECT_ROOT = Path("/kaggle/working/MOESCORE")
DATASET_ROOT = Path("/kaggle/input/datasets/qnv8071/xacle-2027-data/XACLE_Challenge_2027_development_dataset")
WEIGHTS_DATASET_ROOT = Path(
    "/kaggle/input/datasets/qnv8071/xacle-weights"
)
WEIGHTS_ROOT = WEIGHTS_DATASET_ROOT / "weights" / "weights"

HF_CACHE_SOURCE = Path(
    "/kaggle/input/datasets/qnv8071/"
    "moescore-hf-text-cache/hf_hub_cache"
)

HF_HOME = Path(
    "/kaggle/working/moescore/huggingface"
)
UNILM_SOURCE_DIR = None  # None: dùng bản UniLM đã kiểm chứng trong repo
EXISTING_FEATURE_CACHE = None  # Kaggle Input chứa feature_cache.tar + upload_manifest.json, hoặc feature_cache/
WORK_ROOT = Path("/kaggle/working/moescore")
SMOKE_CACHE_ROOT = WORK_ROOT / "smoke"
FEATURE_CACHE_ROOT = WORK_ROOT / "extraction"
RUNS_ROOT = WORK_ROOT / "runs"
CONFIG_ROOT = WORK_ROOT / "configs"
LOG_ROOT = WORK_ROOT / "logs"
HF_HOME = WORK_ROOT / "huggingface"
UPLOAD_DIRECTORY = WORK_ROOT / "upload"
KAGGLE_USERNAME = "qnv8071"
FEATURE_DATASET_SLUG = "moescore-full9-feature-cache"
FEATURE_DATASET_HANDLE = f"{KAGGLE_USERNAME}/{FEATURE_DATASET_SLUG}"
EXPERT_PROFILE = "full_9"
GPU_DEVICES = ["cuda:0", "cuda:1"]
WORKER_DEVICES = {"expert1":"cuda:0", "expert2":"cuda:1", "expert3":"cuda:0",
                  "expert4":"cuda:1", "top2_m2d":"cuda:0", "top2_msclap":"cuda:1",
                  "top2_laion":"cuda:0", "top2_mga":"cuda:1", "m2d_lora":"cuda:0"}
CHECKPOINT_OVERRIDES = {
    "top2_msclap.checkpoint": str(
        WEIGHTS_DATASET_ROOT / "CLAP_weights_2023.pth"
    ),

    "m2d_lora.finetuned_checkpoint": str(
        WEIGHTS_DATASET_ROOT / "m2d-lora.pt"
    ),
}  # Ví dụ: {"m2d_lora.finetuned_checkpoint": "/kaggle/input/.../best.pt"}
EXTRACTION_BATCH_SIZES = {name: 4 for name in WORKER_DEVICES}
SHARD_SIZE = 128
SMOKE_SAMPLES_PER_SPLIT = 4
SMOKE_ONLY = False
RUN_GPU_SMOKE = True
RUN_FULL_EXTRACTION = True
UPLOAD_FEATURE_CACHE = False
RUN_GATE_TRAINING = True
RUN_EVALUATION = True
RUN_TEST_INFERENCE = True
RUN_TEST_EVALUATION = True
INSTALL_MISSING_DEPENDENCIES = True
WHEELHOUSE = None  # Tùy chọn: thư mục wheel nếu tắt Internet khi cài package
GATE_DEVICE = "cuda:0"
TRAIN_BATCH_SIZE = 8
TRAIN_EPOCHS = 50
TRAIN_LEARNING_RATE = 1e-5
SEED = 2027
assert EXPERT_PROFILE == "full_9" and SMOKE_SAMPLES_PER_SPLIT == 4

REBUILD_HF_CACHE = True
UPLOAD_REPAIRED_HF_CACHE = True

HF_CACHE_V2_HANDLE = "qnv8071/moescore-hf-text-cache-v2"

In [2]:
paths = {
    "XACLE dataset": DATASET_ROOT,
    "Weights directory": WEIGHTS_ROOT,
    "HF cache": HF_CACHE_SOURCE,
    "MS-CLAP checkpoint": Path(
        CHECKPOINT_OVERRIDES["top2_msclap.checkpoint"]
    ),
    "M2D-LoRA checkpoint": Path(
        CHECKPOINT_OVERRIDES["m2d_lora.finetuned_checkpoint"]
    ),
}

for name, path in paths.items():
    if path.exists():
        print(f"PASS: {name}")
    else:
        print(f"MISSING: {name} -> {path}")

PASS: XACLE dataset
PASS: Weights directory
PASS: HF cache
PASS: MS-CLAP checkpoint
PASS: M2D-LoRA checkpoint


## 01. Kiểm tra Python và hai GPU

Chấp nhận Python 3.12 hoặc 3.13 và ít nhất hai CUDA device. Stage này không tự chuyển sang CPU hoặc một GPU.


In [3]:
import sys, torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
if sys.version_info[:2] not in {(3,12), (3,13)}:
    raise RuntimeError("Notebook dual-GPU này cần Python 3.12 hoặc 3.13; chọn Kaggle environment tương ứng.")
if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError("Chọn accelerator có 2 GPU (ưu tiên T4 x2) trong Kaggle Settings.")

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA: 13.0
CUDA available: True
GPU count: 2
0 Tesla T4
1 Tesla T4


## 02. Nạp source MOESCORE

Gắn bản source chứa MOE-09.5 hoặc clone commit đã đẩy. Notebook dừng nếu source chưa chứa script hai GPU.


In [4]:
import os, shutil, subprocess, json
for path in (WORK_ROOT, CONFIG_ROOT, LOG_ROOT, RUNS_ROOT): path.mkdir(parents=True, exist_ok=True)
if not PROJECT_ROOT.exists():
    if REPOSITORY_SOURCE is not None:
        shutil.copytree(REPOSITORY_SOURCE, PROJECT_ROOT, symlinks=True)
    else:
        subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(PROJECT_ROOT)], check=True)
required = ["scripts/kaggle_full9_dual_gpu.py", "scripts/kaggle_full9_artifacts.py",
            "scripts/extract_moe_feature_store.py", "scripts/train_moe_gating.py",
            "requirements-kaggle-py312.txt", "requirements-kaggle-py313.txt"]
missing = [name for name in required if not (PROJECT_ROOT/name).is_file()]
if missing: raise FileNotFoundError(f"Source chưa chứa MOE-09.5: {missing}; hãy gắn đúng source hoặc đẩy commit.")
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print("Source:", PROJECT_ROOT)

Cloning into '/kaggle/working/MOESCORE'...


Source: /kaggle/working/MOESCORE


## Check API kaggle 

In [5]:
# import inspect
# import importlib.metadata

# from kaggle import api

# print("Kaggle:", importlib.metadata.version("kaggle"))
# print("Kagglehub:", importlib.metadata.version("kagglehub"))

# api.authenticate()

# for method in ["dataset_metadata", "dataset_download_file"]:
#     fn = getattr(api, method, None)

#     if not callable(fn):
#         raise RuntimeError(f"Missing Kaggle API method: {method}")

#     print("PASS:", method, inspect.signature(fn))

# from scripts.kaggle_full9_artifacts import check_private_upload_target

# handle = "qnv8071/moescore-full9-feature-cache"

# exists = check_private_upload_target(handle)

# print("Dataset exists and is private:", exists)
# print("PASS: Kaggle privacy preflight")

In [6]:
# import json
# import tempfile
# from pathlib import Path
# from kaggle import api

# api.authenticate()

# handles = [
#     "kaggle/titanic",
#     "qnv8071/moescore-hf-text-cache-v2",
#     "qnv8071/moescore-full9-feature-cache",
# ]

# for handle in handles:
#     try:
#         with tempfile.TemporaryDirectory() as tmp:
#             result = api.dataset_metadata(handle, tmp)
#             metadata = json.loads(Path(result).read_text())
#             print(handle, "PASS",
#                   "isPrivate =", metadata.get(
#                       "isPrivate", metadata.get("is_private")
#                   ))
#     except Exception as exc:
#         response = getattr(exc, "response", None)
#         status = getattr(response, "status_code", None)
#         print(handle, "FAIL", "HTTP =", status)

## 03. Phụ thuộc theo phiên bản Python

Giữ bộ PyTorch/CUDA có sẵn. Python 3.12 dùng NumPy 1.26.4 và cài MS-CLAP/LAION-CLAP theo metadata gốc. Python 3.13 giữ chế độ NumPy 2 có kiểm tra smoke riêng; nhánh này vẫn có rủi ro tương thích. Nếu cài đặt đổi package đang được kernel nạp, restart kernel rồi chạy từ stage 00.


In [7]:
import importlib, importlib.metadata as metadata
from scripts.kaggle_full9_dual_gpu import dependency_plan_for_python
requirements_file, legacy_no_deps = dependency_plan_for_python(sys.version_info)
print("Dependency plan:", requirements_file, "legacy metadata bypass:", legacy_no_deps)
loaded_before = {name: metadata.version(name) for name in ("numpy", "scipy", "transformers", "librosa") if name in sys.modules}
if INSTALL_MISSING_DEPENDENCIES:
    command = [sys.executable, str(PROJECT_ROOT/"scripts/install_kaggle_dependencies.py"),
               "--requirements", str(requirements_file),
               "--constraints", str(WORK_ROOT/"torch_constraints.txt"),
               "--report-json", str(LOG_ROOT/"dependencies_install.json")]
    if WHEELHOUSE: command += ["--wheelhouse", str(WHEELHOUSE)]
    subprocess.run(command, check=True)
    if legacy_no_deps:
        legacy = [sys.executable, "-m", "pip", "install", "--no-deps"]
        if WHEELHOUSE: legacy += ["--no-index", "--find-links", str(WHEELHOUSE)]
        subprocess.run(legacy + ["msclap==1.3.4", "laion-clap==1.1.7"], check=True)
    importlib.invalidate_caches()
    changed_loaded = [name for name, old in loaded_before.items() if metadata.version(name) != old]
    if changed_loaded: raise RuntimeError("Các module đang nạp đã đổi phiên bản: " + ", ".join(changed_loaded) + ". Restart kernel rồi chạy lại từ stage 00.")
modules = {"torch":"torch", "torchaudio":"torchaudio", "torchvision":"torchvision",
           "transformers":"transformers", "huggingface_hub":"huggingface-hub",
           "laion_clap":"laion-clap", "msclap":"msclap", "timm":"timm",
           "librosa":"librosa", "torchlibrosa":"torchlibrosa", "nnAudio":"nnAudio",
           "numba":"numba", "soundfile":"soundfile", "scipy":"scipy",
           "einops":"einops", "kagglehub":"kagglehub", "kaggle":"kaggle"}
versions = {}
for module, distribution in modules.items():
    try:
        importlib.import_module(module)
        versions[distribution] = metadata.version(distribution)
    except Exception as exc:
        raise RuntimeError(f"Python {sys.version_info.major}.{sys.version_info.minor} dependency {distribution} import failed: {exc}; cung cấp wheel tương thích và chạy lại kernel.") from exc
if versions["msclap"] != "1.3.4": raise RuntimeError("Yêu cầu msclap==1.3.4")
(LOG_ROOT/"dependency_versions.json").write_text(json.dumps(versions, indent=2))
print(versions)

Dependency plan: /kaggle/working/MOESCORE/requirements-kaggle-py313.txt legacy metadata bypass: True
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 20.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 66.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 16.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 MB 76.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 59.2 MB/s  0:00:00
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9686 sha256=780fb029e33cadc864a3f209b39fc7f8f71db0e77ab0247571bb3542e24f358f
  Stored in directory: /root/.cache/pip/wheels/8

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.68.1 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.68.1 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.68.1 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.68.1 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatib

{
  "status": "PASS",
  "gpu_stack_preserved": {
    "nvidia-cufile": "1.15.1.6",
    "nvidia-curand-cu12": "10.3.10.19",
    "nvidia-nvtx": "13.0.85",
    "torch": "2.11.0+cu130",
    "nvidia-cuda-nvrtc": "13.0.88",
    "nvidia-cuda-nvcc-cu12": "12.9.86",
    "nvidia-cublas": "13.1.0.3",
    "nvidia-cuda-crt": "13.4.92",
    "nvidia-nvjitlink": "13.0.88",
    "nvidia-nvvm": "13.0.88",
    "nvidia-cuda-runtime": "13.0.96",
    "nvidia-ml-py": "13.615.71",
    "nvidia-nvimgcodec-cu13": "0.8.0.22",
    "nvidia-cusolver": "12.0.4.66",
    "nvidia-cusparse": "12.6.3.3",
    "nvidia-nccl-cu12": "2.32.3",
    "torchaudio": "2.11.0+cu130",
    "nvidia-nccl-cu13": "2.28.9",
    "nvidia-cuda-runtime-cu12": "12.9.79",
    "nvidia-nvjitlink-cu12": "12.9.86",
    "nvidia-cusparse-cu12": "12.5.10.65",
    "nvidia-nvshmem-cu13": "3.4.5",
    "nvidia-cuda-cupti-cu12": "12.9.79",
    "nvidia-cufft": "12.0.0.61",
    "torchvision": "0.26.0+cu130",
    "nvidia-cuda-nvrtc-cu12": "12.9.86",
    "nvidia-cu

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
I0000 00:00:1794973277.159887      23 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

... (8 outputs left out)

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

{'torch': '2.11.0+cu130', 'torchaudio': '2.11.0+cu130', 'torchvision': '0.26.0+cu130', 'transformers': '4.57.6', 'huggingface-hub': '0.36.2', 'laion-clap': '1.1.7', 'msclap': '1.3.4', 'timm': '1.0.30', 'librosa': '0.10.2.post1', 'torchlibrosa': '0.1.0', 'nnAudio': '0.3.4', 'numba': '0.68.1', 'soundfile': '0.14.0', 'scipy': '1.16.3', 'einops': '0.8.2', 'kagglehub': '0.4.3', 'kaggle': '1.8.4'}


In [8]:
# ============================================================
# CELL 03.1 — VERIFY DEPENDENCIES & GPU ENVIRONMENT
# ============================================================

import sys
import json
import subprocess
import importlib
import importlib.metadata as metadata
import traceback
from pathlib import Path

import torch

MODULES = {
    "numpy": "numpy",
    "scipy": "scipy",
    "pandas": "pandas",
    "torch": "torch",
    "torchaudio": "torchaudio",
    "torchvision": "torchvision",
    "transformers": "transformers",
    "huggingface_hub": "huggingface-hub",
    "laion_clap": "laion-clap",
    "msclap": "msclap",
    "timm": "timm",
    "librosa": "librosa",
    "torchlibrosa": "torchlibrosa",
    "nnAudio": "nnAudio",
    "numba": "numba",
    "soundfile": "soundfile",
    "einops": "einops",
    "kagglehub": "kagglehub",
    "kaggle": "kaggle",
}

report = {
    "python": sys.version,
    "torch": torch.__version__,
    "cuda_version": torch.version.cuda,
    "gpu_count": torch.cuda.device_count(),
    "dependencies": {},
    "gpu_tests": {},
    "errors": [],
    "warnings": [],
}

print("=" * 65)
print("1. PYTHON ENVIRONMENT")
print("=" * 65)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    report["errors"].append("Unsupported Python version")


print("\n" + "=" * 65)
print("2. DEPENDENCY IMPORT TEST")
print("=" * 65)

for module_name, package_name in MODULES.items():
    try:
        module = importlib.import_module(module_name)
        version = metadata.version(package_name)

        report["dependencies"][package_name] = {
            "status": "PASS",
            "version": version,
        }

        print(f"PASS  {package_name:<22} {version}")

    except Exception as exc:
        report["dependencies"][package_name] = {
            "status": "FAIL",
            "error": f"{type(exc).__name__}: {exc}",
        }

        report["errors"].append(
            f"{package_name}: {type(exc).__name__}: {exc}"
        )

        print(f"FAIL  {package_name:<22} {exc}")


print("\n" + "=" * 65)
print("3. VERSION CHECK")
print("=" * 65)

try:
    msclap_version = metadata.version("msclap")
    assert msclap_version == "1.3.4"
    print("PASS  msclap == 1.3.4")
except Exception as exc:
    report["errors"].append(f"MS-CLAP version: {exc}")
    print("FAIL  MS-CLAP version:", exc)

try:
    numpy_version = metadata.version("numpy")
    print("NumPy version:", numpy_version)

    if sys.version_info[:2] == (3, 13):
        if int(numpy_version.split(".")[0]) < 2:
            report["warnings"].append(
                "Python 3.13 is expected to use NumPy 2 in this pipeline"
            )
except Exception as exc:
    report["errors"].append(f"NumPy version: {exc}")


print("\n" + "=" * 65)
print("4. DUAL GPU TEST")
print("=" * 65)

gpu_count = torch.cuda.device_count()

if not torch.cuda.is_available() or gpu_count < 2:
    report["errors"].append(
        f"Expected 2 CUDA GPUs, found {gpu_count}"
    )
else:
    for i in range(2):
        try:
            device = torch.device(f"cuda:{i}")

            # Small real CUDA computation
            a = torch.randn(256, 256, device=device)
            b = torch.randn(256, 256, device=device)

            result = a @ b

            torch.cuda.synchronize(device)

            assert result.isfinite().all().item()

            free, total = torch.cuda.mem_get_info(i)

            report["gpu_tests"][f"cuda:{i}"] = {
                "status": "PASS",
                "name": torch.cuda.get_device_name(i),
                "total_vram_gb": round(total / 1024**3, 2),
                "free_vram_gb": round(free / 1024**3, 2),
            }

            print(
                f"PASS  GPU {i}: {torch.cuda.get_device_name(i)} "
                f"| Free: {free / 1024**3:.2f} GB"
            )

            del a, b, result

        except Exception as exc:
            report["gpu_tests"][f"cuda:{i}"] = {
                "status": "FAIL",
                "error": str(exc),
            }
            report["errors"].append(f"GPU {i}: {exc}")
            print(f"FAIL  GPU {i}: {exc}")


print("\n" + "=" * 65)
print("5. PIP DEPENDENCY CONSISTENCY")
print("=" * 65)

check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True,
)

report["pip_check"] = {
    "returncode": check.returncode,
    "output": check.stdout + check.stderr,
}

if check.returncode == 0:
    print("PASS  pip check")
else:
    print("WARNING  pip check reported dependency conflicts:")
    print((check.stdout + check.stderr)[:3000])

    report["warnings"].append(
        "pip check failed; inspect dependency conflicts"
    )


print("\n" + "=" * 65)
print("6. FINAL RESULT")
print("=" * 65)

report["status"] = (
    "PASS" if not report["errors"] else "FAIL"
)

output_path = Path(
    "/kaggle/working/dependency_test_report.json"
)

output_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

print("Status:", report["status"])
print("Errors:", len(report["errors"]))
print("Warnings:", len(report["warnings"]))
print("Report:", output_path)

if report["errors"]:
    print("\nFAILED CHECKS:")
    for error in report["errors"]:
        print(" -", error)

    raise RuntimeError(
        "Dependency verification FAILED. See report above."
    )

print("\nAll import and GPU checks PASSED!")
print("Next stage: Kaggle Input Discovery / Offline Preflight")

1. PYTHON ENVIRONMENT
Python: 3.13.16
PyTorch: 2.11.0+cu130
CUDA: 13.0

2. DEPENDENCY IMPORT TEST
PASS  numpy                  2.1.3
PASS  scipy                  1.16.3
PASS  pandas                 2.3.3
PASS  torch                  2.11.0+cu130
PASS  torchaudio             2.11.0+cu130
PASS  torchvision            0.26.0+cu130
PASS  transformers           4.57.6
PASS  huggingface-hub        0.36.2
PASS  laion-clap             1.1.7
PASS  msclap                 1.3.4
PASS  timm                   1.0.30
PASS  librosa                0.10.2.post1
PASS  torchlibrosa           0.1.0
PASS  nnAudio                0.3.4
PASS  numba                  0.68.1
PASS  soundfile              0.14.0
PASS  einops                 0.8.2
PASS  kagglehub              0.4.3
PASS  kaggle                 1.8.4

3. VERSION CHECK
PASS  msclap == 1.3.4
NumPy version: 2.1.3

4. DUAL GPU TEST
PASS  GPU 0: Tesla T4 | Free: 14.41 GB
PASS  GPU 1: Tesla T4 | Free: 14.41 GB

5. PIP DEPENDENCY CONSISTENCY
WARNING  pip ch

## 04. Khám phá Kaggle Inputs

Kiểm tra dataset, trọng số, HF cache và UniLM. Không suy diễn tên mount checkpoint.


In [9]:
from scripts import kaggle_moe_pipeline as pipeline
print("Kaggle inputs:", [str(p) for p in sorted(Path("/kaggle/input").iterdir())])
if not DATASET_ROOT.exists():
    candidates = sorted({p.parent.parent for p in Path("/kaggle/input").rglob("meta_data/train_average.csv")})
    if len(candidates) != 1: raise FileNotFoundError(f"Chọn DATASET_ROOT chính xác; candidates={candidates}")
    DATASET_ROOT = candidates[0]
for split in ("train", "validation", "test"):
    for path in (DATASET_ROOT/f"meta_data/{split}_average.csv", DATASET_ROOT/f"wav/{split}"):
        if not path.exists(): raise FileNotFoundError(path)
if not WEIGHTS_ROOT.is_dir(): raise FileNotFoundError(f"Chỉnh WEIGHTS_ROOT: {WEIGHTS_ROOT}")
if not HF_CACHE_SOURCE.exists(): raise FileNotFoundError(f"Chỉnh HF_CACHE_SOURCE: {HF_CACHE_SOURCE}")
print("Dataset:", DATASET_ROOT, "Weights:", WEIGHTS_ROOT, "HF source:", HF_CACHE_SOURCE)

Kaggle inputs: ['/kaggle/input/datasets']
Dataset: /kaggle/input/datasets/qnv8071/xacle-2027-data/XACLE_Challenge_2027_development_dataset Weights: /kaggle/input/datasets/qnv8071/xacle-weights/weights/weights HF source: /kaggle/input/datasets/qnv8071/moescore-hf-text-cache/hf_hub_cache



## STAGE 04A — REBUILD, VERIFY, PACKAGE AND UPLOAD HF CACHE


In [10]:
import os
import sys
import json
import shutil
import tarfile
import hashlib
import subprocess

from pathlib import Path

if REBUILD_HF_CACHE:

    print("=" * 65)
    print("STAGE 04A — REBUILD HUGGING FACE CACHE")
    print("=" * 65)

    models = [
        "roberta-base",
        "bert-base-uncased",
        "google-bert/bert-base-uncased",
        "gpt2",
        "facebook/bart-base",
    ]

    hf_hub = HF_HOME / "hub"
    hf_hub.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # A. REMOVE CORRUPTED CACHE FROM WORKING DIRECTORY
    # --------------------------------------------------------

    for model_id in models:
        folder = hf_hub / (
            "models--" + model_id.replace("/", "--")
        )

        if folder.exists():
            shutil.rmtree(folder)
            print("Removed corrupted cache:", model_id)

    # --------------------------------------------------------
    # B. DOWNLOAD FOUR MODELS
    # --------------------------------------------------------

    online_env = os.environ.copy()

    online_env.update({
        "HF_HOME": str(HF_HOME),
        "HF_HUB_CACHE": str(hf_hub),
        "HF_HUB_OFFLINE": "0",
        "TRANSFORMERS_OFFLINE": "0",
    })

    online_env.pop("TRANSFORMERS_CACHE", None)

    download_code = r"""
import sys
from huggingface_hub import snapshot_download

cache_dir = sys.argv[1]
model_ids = sys.argv[2:]

for model_id in model_ids:

    print("Downloading:", model_id, flush=True)

    path = snapshot_download(
        repo_id=model_id,
        cache_dir=cache_dir,
        allow_patterns=[
            "*.json",
            "*.txt",
            "*.model",
            "*.safetensors",
            "pytorch_model.bin",
        ],
        local_files_only=False,
        force_download=True,
        max_workers=4,
    )

    print("Downloaded:", model_id, path, flush=True)
"""

    subprocess.run(
        [
            sys.executable,
            "-c",
            download_code,
            str(hf_hub),
            *models,
        ],
        env=online_env,
        check=True,
    )

    # --------------------------------------------------------
    # C. VERIFY SNAPSHOTS
    # --------------------------------------------------------

    for model_id in models:

        folder = hf_hub / (
            "models--" + model_id.replace("/", "--")
        )

        snapshots = folder / "snapshots"

        assert snapshots.is_dir(), (
            f"Missing snapshots: {model_id}"
        )

        files = [
            p for p in snapshots.rglob("*")
            if p.is_file() or p.is_symlink()
        ]

        assert files, f"Empty snapshot: {model_id}"

        invalid_files = [
            p for p in files
            if not p.exists() or p.stat().st_size == 0
        ]

        assert not invalid_files, (
            f"Invalid files for {model_id}: "
            f"{invalid_files[:5]}"
        )

        print(
            f"PASS: {model_id} — {len(files)} valid files"
        )

    # --------------------------------------------------------
    # D. RUN ORIGINAL MOESCORE OFFLINE PREFLIGHT
    # --------------------------------------------------------

    offline_env = online_env.copy()

    offline_env["HF_HUB_OFFLINE"] = "1"
    offline_env["TRANSFORMERS_OFFLINE"] = "1"

    verify = subprocess.run(
        [
            sys.executable,
            str(
                PROJECT_ROOT
                / "scripts/prepare_moe_text_cache.py"
            ),
            "--cache-dir",
            str(HF_HOME),
            "--profile",
            "full_9",
        ],
        env=offline_env,
        capture_output=True,
        text=True,
    )

    print("\n=== OFFLINE PREFLIGHT ===")
    print(verify.stdout)
    print(verify.stderr)

    if verify.returncode != 0:
        raise RuntimeError(
            "Hugging Face offline preflight FAILED"
        )

    print("PASS: All four HF models verified offline")

    # --------------------------------------------------------
    # E. PACKAGE AS TAR TO PRESERVE SYMLINKS
    # --------------------------------------------------------

    upload_dir = WORK_ROOT / "hf_cache_v2_upload"
    upload_dir.mkdir(parents=True, exist_ok=True)

    archive_path = upload_dir / "hf_hub_cache.tar"

    with tarfile.open(
        archive_path,
        mode="w",
        dereference=False,
    ) as tar:

        for model_id in models:

            folder_name = (
                "models--" + model_id.replace("/", "--")
            )

            source = hf_hub / folder_name

            tar.add(
                source,
                arcname=f"hub/{folder_name}",
                recursive=True,
            )

    # SHA-256
    digest = hashlib.sha256()

    with archive_path.open("rb") as stream:
        for chunk in iter(
            lambda: stream.read(8 * 1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    # Confirm archive contains all four model trees.
    with tarfile.open(archive_path, "r") as tar:
        member_names = [m.name for m in tar.getmembers()]

        for model_id in models:
            prefix = (
                "hub/models--"
                + model_id.replace("/", "--")
            )

            assert any(
                name == prefix or name.startswith(prefix + "/")
                for name in member_names
            ), f"Archive missing: {model_id}"

    manifest = {
        "archive": archive_path.name,
        "sha256": digest.hexdigest(),
        "models": models,
        "offline_preflight": "PASS",
    }

    (upload_dir / "manifest.json").write_text(
        json.dumps(manifest, indent=2),
        encoding="utf-8",
    )

    print("\nArchive:", archive_path)
    print(
        "Archive size GiB:",
        round(archive_path.stat().st_size / 1024**3, 2),
    )
    print("SHA-256:", manifest["sha256"])

    # --------------------------------------------------------
    # F. UPLOAD PRIVATE KAGGLE DATASET V2
    # --------------------------------------------------------

    if UPLOAD_REPAIRED_HF_CACHE:

        import kagglehub

        print(
            "\nUploading:",
            HF_CACHE_V2_HANDLE,
        )

        kagglehub.dataset_upload(
            handle=HF_CACHE_V2_HANDLE,
            local_dataset_dir=str(upload_dir),
            version_notes=(
                "Rebuilt MOESCORE full_9 HF cache "
                "with verified offline resources "
                "and preserved symlinks."
            ),
        )

        print("\nUPLOAD COMPLETED")
        print(
            "https://www.kaggle.com/datasets/"
            + HF_CACHE_V2_HANDLE
        )

    print("\nSTAGE 04A COMPLETED SUCCESSFULLY")

else:

    print("STAGE 04A SKIPPED — using attached HF cache")

STAGE 04A — REBUILD HUGGING FACE CACHE
Downloading: roberta-base


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Downloaded: roberta-base /kaggle/working/moescore/huggingface/hub/models--roberta-base/snapshots/e2da8e2f811d1448a5b465c236feacd80ffbac7b
Downloading: bert-base-uncased


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Downloaded: bert-base-uncased /kaggle/working/moescore/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594
Downloading: google-bert/bert-base-uncased


Fetching 15 files:   7%|▋         | 1/15 [00:00<00:01, 10.00it/s]

... (6 outputs left out)

Upload successful: /kaggle/working/moescore/hf_cache_v2_upload/manifest.json (284B)
Starting upload for file /kaggle/working/moescore/hf_cache_v2_upload/hf_hub_cache.tar



Uploading: 100%|██████████| 4.99G/4.99G [01:26<00:00, 57.7MB/s]

Upload successful: /kaggle/working/moescore/hf_cache_v2_upload/hf_hub_cache.tar (5GB)


Your dataset version has been created.
Files are being processed...
See at: https://api.kaggle.com/datasets/qnv8071/moescore-hf-text-cache-v2

UPLOAD COMPLETED
https://www.kaggle.com/datasets/qnv8071/moescore-hf-text-cache-v2

STAGE 04A COMPLETED SUCCESSFULLY


## 05. Khôi phục Hugging Face cache

Giữ nguyên cây Hub `blobs/refs/snapshots` và symlink tương đối. Dùng cache đã gắn hoặc tar archive, không tải model khi trích xuất.


In [11]:
import tarfile
hub = HF_HOME/"hub"
if not hub.is_dir():
    if HF_CACHE_SOURCE.is_dir():
        candidate = HF_CACHE_SOURCE/"hub" if (HF_CACHE_SOURCE/"hub").is_dir() else HF_CACHE_SOURCE
        if not any(candidate.glob("models--*")): raise ValueError("HF cache source thiếu models--* trong hub")
        shutil.copytree(candidate, hub, symlinks=True)
    else:
        HF_HOME.mkdir(parents=True, exist_ok=True)
        with tarfile.open(HF_CACHE_SOURCE) as archive:
            archive.extractall(HF_HOME, filter="data")
        if not hub.is_dir(): raise ValueError("HF archive phải chứa thư mục hub/ nguyên cấu trúc")
for name in ("HF_HOME", "HF_HUB_CACHE", "TRANSFORMERS_CACHE"):
    os.environ[name] = str(HF_HOME if name == "HF_HOME" else hub)
for name in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"):
    os.environ[name] = "1"
print("Hub cache:", hub, "models:", [p.name for p in hub.glob("models--*")])

Hub cache: /kaggle/working/moescore/huggingface/hub models: ['models--roberta-base', 'models--facebook--bart-base', 'models--bert-base-uncased', 'models--google-bert--bert-base-uncased', 'models--gpt2']


## 06. Tạo cấu hình full_9

Tạo bản YAML trong /kaggle/working. Tùy chọn `CHECKPOINT_OVERRIDES` dùng tên expert.field; không sửa source.


In [12]:
from scripts.extract_moe_feature_store import FULL9_CHECKPOINT_FIELDS
from model.expert_adapters.registry import EXPERT_PROFILES
from scripts.kaggle_full9_dual_gpu import validate_worker_devices
from scripts import kaggle_full9_artifacts as artifacts
import yaml
order = EXPERT_PROFILES["full_9"]
validate_worker_devices(WORKER_DEVICES, order)
defaults = {
 "expert1":{"pretrained_checkpoint":"pretrained/laion-clap.pt","finetuned_checkpoint":"finetuned_weights/expert1_best.pt"},
 "expert2":{"pretrained_checkpoint":"pretrained/mga-clap.pt","finetuned_checkpoint":"finetuned_weights/expert2_best.pt"},
 "expert3":{"base_checkpoint":"pretrained/checkpoint-30.pth","finetuned_checkpoint":"finetuned_weights/expert3_best.pt"},
 "expert4":{"beats_checkpoint":"pretrained/BEATs_iter3_plus_AS2M.pt","clap_checkpoint":"pretrained/laion-clap.pt","finetuned_checkpoint":"finetuned_weights/expert4_best.pt"},
 "top2_m2d":{"checkpoint":"pretrained/checkpoint-30.pth"},
 "top2_msclap":{"checkpoint":"pretrained/CLAP_weights_2023.pth"},
 "top2_laion":{"checkpoint":"pretrained/laion-clap.pt"},
 "top2_mga":{"pretrained_checkpoint":"pretrained/mga-clap.pt"},
 "m2d_lora":{"base_checkpoint":"pretrained/checkpoint-30.pth","finetuned_checkpoint":"best.pt"}}
for key, path in CHECKPOINT_OVERRIDES.items():
    expert, field = key.split(".", 1)
    if expert not in defaults or field not in defaults[expert]: raise ValueError(f"Unknown checkpoint override: {key}")
    defaults[expert][field] = str(path)
expert_config, gate_config, resolved = pipeline.generate_configs(
    PROJECT_ROOT, DATASET_ROOT, WEIGHTS_ROOT, HF_HOME, FEATURE_CACHE_ROOT, RUNS_ROOT,
    CONFIG_ROOT, expert_checkpoints=defaults, profile="full_9", unilm_source_dir=UNILM_SOURCE_DIR)
config = yaml.safe_load(gate_config.read_text())
config["runtime"].update(device=GATE_DEVICE, seed=SEED)
config["training"].update(batch_size=TRAIN_BATCH_SIZE, max_epochs=TRAIN_EPOCHS, learning_rate=TRAIN_LEARNING_RATE)
gate_config.write_text(yaml.safe_dump(config, sort_keys=False))
(CONFIG_ROOT/"worker_devices.json").write_text(json.dumps(WORKER_DEVICES, indent=2))
print("Expert config:", expert_config, "Gate config:", gate_config)

Expert config: /kaggle/working/moescore/configs/moe_expert_config.yaml Gate config: /kaggle/working/moescore/configs/moe_gating_config.yaml


## 07. Offline preflight

Kiểm chứng đủ 9 checkpoint, cấu trúc dataset, source UniLM, import dependency và snapshot text model trước khi nạp expert.


In [13]:
from configs.moe_config import load_moe_config, resolve_moe_paths
preflight_report = pipeline.preflight(resolve_moe_paths(load_moe_config(expert_config), expert_config), DATASET_ROOT,
                                      WORK_ROOT/"preflight", require_cuda=True, check_cache=True)
if preflight_report["status"] != "PASS": raise RuntimeError(preflight_report)
subprocess.run([sys.executable, str(PROJECT_ROOT/"scripts/prepare_moe_text_cache.py"),
                "--cache-dir", str(HF_HOME), "--profile", "full_9"], check=True)
print("Preflight PASS:", preflight_report["splits"])

# Extra offline preflight for LAION-CLAP's import-time BART dependency

import os
import sys
import subprocess

bart_check = r"""
from transformers import BartTokenizer

tok = BartTokenizer.from_pretrained(
    "facebook/bart-base",
    local_files_only=True,
)

assert tok.vocab_size > 0
print("PASS: facebook/bart-base tokenizer", tok.vocab_size)

# Test the exact import that previously failed.
import laion_clap
print("PASS: laion_clap offline import")
"""

offline_env = os.environ.copy()
offline_env["HF_HOME"] = str(HF_HOME)
offline_env["HF_HUB_CACHE"] = str(HF_HOME / "hub")
offline_env["HF_HUB_OFFLINE"] = "1"
offline_env["TRANSFORMERS_OFFLINE"] = "1"

subprocess.run(
    [sys.executable, "-c", bart_check],
    env=offline_env,
    cwd=PROJECT_ROOT,
    check=True,
)

print("PASS: LAION-CLAP dependencies verified offline")

/usr/local/lib/python3.13/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


{
  "status": "PASS",
  "cache_dir": "/kaggle/working/moescore/huggingface/hub",
  "model_ids": [
    "roberta-base",
    "bert-base-uncased",
    "google-bert/bert-base-uncased",
    "gpt2"
  ],
  "resources": [
    {
      "model_id": "roberta-base",
      "model_type": "roberta",
      "tokenizer_class": "RobertaTokenizerFast",
      "weights_required": true
    },
    {
      "model_id": "bert-base-uncased",
      "model_type": "bert",
      "tokenizer_class": "BertTokenizer",
      "weights_required": false
    },
    {
      "model_id": "google-bert/bert-base-uncased",
      "model_type": "bert",
      "tokenizer_class": "BertTokenizerFast",
      "weights_required": true
    },
    {
      "model_id": "gpt2",
      "model_type": "gpt2",
      "tokenizer_class": "GPT2TokenizerFast",
      "weights_required": true
    }
  ],
  "offline_extraction": true,
  "model_construction": false
}


/usr/local/lib/python3.13/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


{
  "status": "PASS",
  "cache_dir": "/kaggle/working/moescore/huggingface/hub",
  "model_ids": [
    "roberta-base",
    "bert-base-uncased",
    "google-bert/bert-base-uncased",
    "gpt2"
  ],
  "resources": [
    {
      "model_id": "roberta-base",
      "model_type": "roberta",
      "tokenizer_class": "RobertaTokenizerFast",
      "weights_required": true
    },
    {
      "model_id": "bert-base-uncased",
      "model_type": "bert",
      "tokenizer_class": "BertTokenizer",
      "weights_required": false
    },
    {
      "model_id": "google-bert/bert-base-uncased",
      "model_type": "bert",
      "tokenizer_class": "BertTokenizerFast",
      "weights_required": true
    },
    {
      "model_id": "gpt2",
      "model_type": "gpt2",
      "tokenizer_class": "GPT2TokenizerFast",
      "weights_required": true
    }
  ],
  "offline_extraction": true,
  "model_construction": false
}
Preflight PASS: {'train': {'count': 7500, 'headers': ['wav_file_name', 'text', 'average_score', 

/usr/local/lib/python3.13/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
I0000 00:00:1794973533.161821     279 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


PASS: facebook/bart-base tokenizer 50265
PASS: laion_clap offline import
PASS: LAION-CLAP dependencies verified offline


## 08. Smoke trích xuất thực trên hai GPU

4 mẫu mỗi split, cả 9 expert; mỗi GPU chỉ giữ một model. Log từng expert trong `smoke/logs/`. Dừng nếu có worker lỗi.


In [14]:
def run_cli(*parts):
    command = [sys.executable, *map(str, parts)]
    print("RUN", " ".join(command), flush=True)
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
if RUN_GPU_SMOKE:
    smoke_report = LOG_ROOT/"smoke_extraction.json"
    run_cli(PROJECT_ROOT/"scripts/kaggle_full9_dual_gpu.py", "--config", expert_config,
            "--dataset-root", DATASET_ROOT, "--output-root", SMOKE_CACHE_ROOT,
            "--worker-devices-json", CONFIG_ROOT/"worker_devices.json", "--smoke",
            "--batch-sizes", ",".join(str(EXTRACTION_BATCH_SIZES[n]) for n in order),
            "--shard-size", SHARD_SIZE, "--report-json", smoke_report)
    smoke_result = json.loads(smoke_report.read_text())
    if smoke_result["status"] != "PASS": raise RuntimeError(smoke_result)
    artifacts.validate_full9_store(smoke_result["feature_store_root"], {s:4 for s in ("train","validation","test")})
    print("Smoke GPU assignment:", smoke_result["worker_devices"])
    for name, info in smoke_result.get("workers", {}).items():
        print(name, info["device"], info["gpu_name"], "peak allocated MiB",
              round(info["peak_allocated_vram_bytes"] / 2**20, 1), "loaded", info["model_constructed"])
else:
    raise RuntimeError("Full execution requires RUN_GPU_SMOKE=True or an explicitly verified prior smoke stage")

RUN /usr/bin/python3 /kaggle/working/MOESCORE/scripts/kaggle_full9_dual_gpu.py --config /kaggle/working/moescore/configs/moe_expert_config.yaml --dataset-root /kaggle/input/datasets/qnv8071/xacle-2027-data/XACLE_Challenge_2027_development_dataset --output-root /kaggle/working/moescore/smoke --worker-devices-json /kaggle/working/moescore/configs/worker_devices.json --smoke --batch-sizes 4,4,4,4,4,4,4,4,4 --shard-size 128 --report-json /kaggle/working/moescore/logs/smoke_extraction.json


Identity train: 100%|██████████| 4/4 [00:00<00:00, 10.76it/s]


train: 4 pairs; columns=['wav_file_name', 'text', 'average_score', 'model_ID', 'model_name']; labels_consumed=True


Identity test:  50%|█████     | 2/4 [00:00<00:00, 12.69it/s]

validation: 4 pairs; columns=['wav_file_name', 'text', 'average_score', 'model_ID', 'model_name']; labels_consumed=True


Identity test: 100%|██████████| 4/4 [00:00<00:00, 13.78it/s]


test: 4 pairs; columns=['wav_file_name', 'text', 'average_score', 'model_ID', 'model_name']; labels_consumed=False
{
  "status": "PASS",
  "assembly_id": "40807a0f71dd0f3f9332fd3eb52c5f93d1aeb6c9da663924b73e368120a1c2f4",
  "feature_store_root": "/kaggle/working/moescore/smoke/feature_cache/assemblies/full_9/40807a0f71dd0f3f9332fd3eb52c5f93d1aeb6c9da663924b73e368120a1c2f4",
  "worker_devices": {
    "expert1": "cuda:0",
    "expert2": "cuda:1",
    "expert3": "cuda:0",
    "expert4": "cuda:1",
    "top2_m2d": "cuda:0",
    "top2_msclap": "cuda:1",
    "top2_laion": "cuda:0",
    "top2_mga": "cuda:1",
    "m2d_lora": "cuda:0"
  },
  "gpu_names": {
    "cuda:0": "Tesla T4",
    "cuda:1": "Tesla T4"
  },
  "workers": {
    "expert1": {
      "status": "PASS",
      "expert": "expert1",
      "device": "cuda:0",
      "gpu_name": "Tesla T4",
      "checkpoint_provenance": {
        "finetuned_checkpoint": {
          "sha256": "e2f5f1183c36fe0db0eb2a89d163023335e9f157f89e43c14018c3193c5725

In [15]:
import json
from pathlib import Path
if RUN_GPU_SMOKE:
    log_root = Path("/kaggle/working/moescore/smoke/logs")
    
    for stage_file in sorted(log_root.glob("*/stage.json")):
        report = json.loads(stage_file.read_text())
    
        print("Job:", stage_file.parent.name)
        print("Status:", report.get("status"))
        print("Assembly ID:", report.get("assembly_id"))
        print("Feature store:", report.get("feature_store_root"))
        print("Splits:", report.get("splits"))

Job: 40807a0f71dd0f3f9332fd3eb52c5f93d1aeb6c9da663924b73e368120a1c2f4
Status: PASS
Assembly ID: 40807a0f71dd0f3f9332fd3eb52c5f93d1aeb6c9da663924b73e368120a1c2f4
Feature store: /kaggle/working/moescore/smoke/feature_cache/assemblies/full_9/40807a0f71dd0f3f9332fd3eb52c5f93d1aeb6c9da663924b73e368120a1c2f4
Splits: {'test': {'rows': 4, 'store_identity': '092dcda3d7e3f958bf57e7e1d854d7c0286f1e4c37c9e335ee845bb7c776305c'}, 'train': {'rows': 4, 'store_identity': 'bc492875355ff2586ffdb40ea3a8bbcb57f80af085c6821f32922bf914cf7028'}, 'validation': {'rows': 4, 'store_identity': 'f16e2c0985339b1b34b5dcf83e181e5becbbf6ad5db73c092e4a02449e69e4c7'}}


## 09. Smoke gate trên đặc trưng thật

Ba bước optimizer với feature store vừa trích xuất; không nạp chín expert cùng lúc.


In [16]:
if RUN_GPU_SMOKE:
    smoke_gate = pipeline.smoke_store(smoke_result["feature_store_root"], config)
    if smoke_gate["status"] != "PASS" or smoke_gate["optimizer_steps"] != 3: raise RuntimeError(smoke_gate)
    (LOG_ROOT/"smoke_gate.json").write_text(json.dumps(smoke_gate, indent=2))
    print("Gate smoke:", smoke_gate)

Gate smoke: {'status': 'PASS', 'scope': 'SMOKE_ONLY', 'sample_count': 4, 'optimizer_steps': 3, 'source': 'REAL_CHECKPOINT_EXTRACTION', 'gate_device': 'cpu', 'steps': [{'total_loss': 0.016585882753133774, 'regression_loss': 0.008117134682834148, 'diversity_loss': 0.042343735694885254, 'gradient_norm': 1.5949829816818237}, {'total_loss': 0.05767638236284256, 'regression_loss': 0.049457088112831116, 'diversity_loss': 0.04109647870063782, 'gradient_norm': 2.894866466522217}, {'total_loss': 0.012789364904165268, 'regression_loss': 0.008310540579259396, 'diversity_loss': 0.022394120693206787, 'gradient_norm': 1.3123509883880615}], 'full_training_completed': False}


## 10. Trích xuất full_9 hai GPU

Chỉ chạy khi `SMOKE_ONLY=False` và `RUN_FULL_EXTRACTION=True`. Job hash checkpoint/sample một lần, hai queue GPU chạy song song, sau đó assemble.


In [17]:
if not SMOKE_ONLY and RUN_FULL_EXTRACTION:
    final_report = LOG_ROOT/"full_extraction.json"
    run_cli(PROJECT_ROOT/"scripts/kaggle_full9_dual_gpu.py", "--config", expert_config,
            "--dataset-root", DATASET_ROOT, "--output-root", FEATURE_CACHE_ROOT,
            "--worker-devices-json", CONFIG_ROOT/"worker_devices.json",
            "--batch-sizes", ",".join(str(EXTRACTION_BATCH_SIZES[n]) for n in order),
            "--shard-size", SHARD_SIZE, "--report-json", final_report)
    full_result = json.loads(final_report.read_text())
    if full_result["status"] != "PASS": raise RuntimeError(full_result)
elif SMOKE_ONLY:
    print("SMOKE_ONLY: bỏ qua trích xuất đầy đủ")

RUN /usr/bin/python3 /kaggle/working/MOESCORE/scripts/kaggle_full9_dual_gpu.py --config /kaggle/working/moescore/configs/moe_expert_config.yaml --dataset-root /kaggle/input/datasets/qnv8071/xacle-2027-data/XACLE_Challenge_2027_development_dataset --output-root /kaggle/working/moescore/extraction --worker-devices-json /kaggle/working/moescore/configs/worker_devices.json --batch-sizes 4,4,4,4,4,4,4,4,4 --shard-size 128 --report-json /kaggle/working/moescore/logs/full_extraction.json


Identity train: 100%|██████████| 7500/7500 [09:44<00:00, 12.83it/s]


train: 7500 pairs; columns=['wav_file_name', 'text', 'average_score', 'model_ID', 'model_name']; labels_consumed=True


Identity validation: 100%|██████████| 3000/3000 [03:42<00:00, 13.48it/s]


validation: 3000 pairs; columns=['wav_file_name', 'text', 'average_score', 'model_ID', 'model_name']; labels_consumed=True


Identity test: 100%|██████████| 3000/3000 [03:58<00:00, 12.58it/s]


test: 3000 pairs; columns=['wav_file_name', 'text', 'average_score', 'model_ID', 'model_name']; labels_consumed=False
{
  "status": "PASS",
  "assembly_id": "acde0938c1af064a6742e545ee6196c45f5cf95227f55233c48021db31045990",
  "feature_store_root": "/kaggle/working/moescore/extraction/feature_cache/assemblies/full_9/acde0938c1af064a6742e545ee6196c45f5cf95227f55233c48021db31045990",
  "worker_devices": {
    "expert1": "cuda:0",
    "expert2": "cuda:1",
    "expert3": "cuda:0",
    "expert4": "cuda:1",
    "top2_m2d": "cuda:0",
    "top2_msclap": "cuda:1",
    "top2_laion": "cuda:0",
    "top2_mga": "cuda:1",
    "m2d_lora": "cuda:0"
  },
  "gpu_names": {
    "cuda:0": "Tesla T4",
    "cuda:1": "Tesla T4"
  },
  "workers": {
    "expert1": {
      "status": "PASS",
      "expert": "expert1",
      "device": "cuda:0",
      "gpu_name": "Tesla T4",
      "checkpoint_provenance": {
        "finetuned_checkpoint": {
          "sha256": "e2f5f1183c36fe0db0eb2a89d163023335e9f157f89e43c14018c3

## 11. Xác thực feature store

Kiểm tra SHA-256, profile, thứ tự, số dòng 7500/3000/3000 và width 27399. Cache cũ chỉ được dùng lại khi job provenance trùng.


In [18]:
if not SMOKE_ONLY:
    if not RUN_FULL_EXTRACTION and not EXISTING_FEATURE_CACHE:
        raise ValueError("Bật RUN_FULL_EXTRACTION hoặc cung cấp EXISTING_FEATURE_CACHE đã xác thực")
    if EXISTING_FEATURE_CACHE and not RUN_FULL_EXTRACTION:
        mounted = Path(EXISTING_FEATURE_CACHE)
        if not (FEATURE_CACHE_ROOT/"feature_cache").exists():
            if (mounted/"upload_manifest.json").is_file(): artifacts.restore_feature_cache(mounted, FEATURE_CACHE_ROOT)
            elif (mounted/"feature_cache").is_dir(): shutil.copytree(mounted/"feature_cache", FEATURE_CACHE_ROOT/"feature_cache", symlinks=True)
            else: raise FileNotFoundError("Existing cache thiếu archive hoặc feature_cache/")
        # Chỉ tin cache khi source/checkpoints và identity hiện tại tái lập đúng.
        from scripts.extract_moe_feature_store import prepare_job, completed_stores_match, assembly_store_root
        _, expected_job, expected_manifests = prepare_job(expert_config, DATASET_ROOT, FEATURE_CACHE_ROOT,
            batch_sizes=EXTRACTION_BATCH_SIZES, shard_size=SHARD_SIZE, profile="full_9", worker_devices=WORKER_DEVICES)
        if not completed_stores_match(expected_job, expected_manifests, FEATURE_CACHE_ROOT):
            raise ValueError("Existing feature cache incomplete; bật RUN_FULL_EXTRACTION để resume")
        full_result = {"status":"PASS", "feature_store_root":str(assembly_store_root(FEATURE_CACHE_ROOT, expected_job)),
                       "assembly_id":__import__("scripts.extract_moe_feature_store", fromlist=["job_identity"]).job_identity(expected_job)}
    full_store = Path(full_result["feature_store_root"])
    store_validation = artifacts.validate_full9_store(full_store, {"train":7500,"validation":3000,"test":3000})
    config["data"]["feature_store_root"] = str(full_store)
    gate_config.write_text(yaml.safe_dump(config, sort_keys=False))
    (LOG_ROOT/"full_store_validation.json").write_text(json.dumps(store_validation, indent=2))
    print(store_validation)

{'train': {'samples': 7500, 'store_identity': '57af12ba01c616e5affb4d11d17d6e4364ac696dbec4d46d28b558d86a212d4c'}, 'validation': {'samples': 3000, 'store_identity': '2e68ff937bb6e9e05156ec429fb8ec091a73c3be7c368895f6ae9bb4e6e18168'}, 'test': {'samples': 3000, 'store_identity': '95818985aa329dc33809246d283c2b322592f22f32ea1c7eb4e73af509aedebb'}}


## 12. Upload cache lên Kaggle Dataset

Chỉ upload `feature_cache.tar` và manifest SHA. `kagglehub.dataset_upload` tạo dataset hoặc version; sau upload tải lại manifest để kiểm chứng. Dừng trước training nếu upload lỗi. Dataset mới giữ private mặc định.


In [19]:
if not SMOKE_ONLY:
    if UPLOAD_FEATURE_CACHE:
        package = artifacts.package_feature_cache(FEATURE_CACHE_ROOT, UPLOAD_DIRECTORY, stores=store_validation)
        upload_report = artifacts.upload_feature_cache(FEATURE_DATASET_HANDLE, UPLOAD_DIRECTORY,
            version_notes=f"MOESCORE full_9 {full_result['assembly_id']}")
        (LOG_ROOT/"upload.json").write_text(json.dumps(upload_report, indent=2))
        print("Uploaded:", upload_report["url"])
    elif RUN_GATE_TRAINING:
        print("UPLOAD_FEATURE_CACHE=False được chọn rõ ràng; tiếp tục training trên cache cục bộ")

UPLOAD_FEATURE_CACHE=False được chọn rõ ràng; tiếp tục training trên cache cục bộ


## 13. Train Gating Network

Chỉ học gate 27399→512→9. Dùng feature store, không nạp expert. Resume từ `latest_gate.pt` nếu đã có.


In [20]:
if not SMOKE_ONLY and RUN_GATE_TRAINING:
    latest = RUNS_ROOT/"latest_gate.pt"
    args = [PROJECT_ROOT/"scripts/train_moe_gating.py", "--config", gate_config,
            "--device", GATE_DEVICE, "--report-json", LOG_ROOT/"training.json"]
    if latest.is_file(): args += ["--resume", latest]
    run_cli(*args)
    train_report = json.loads((LOG_ROOT/"training.json").read_text())
    if train_report["status"] != "PASS": raise RuntimeError(train_report)
    print("Best:", train_report.get("best_checkpoint"), "Latest:", train_report.get("latest_checkpoint"))

RUN /usr/bin/python3 /kaggle/working/MOESCORE/scripts/train_moe_gating.py --config /kaggle/working/moescore/configs/moe_gating_config.yaml --device cuda:0 --report-json /kaggle/working/moescore/logs/training.json


Epoch 1: step=937, train_loss=0.137027, validation={'score_scale': 'raw_0_to_10', 'sample_count': 3000, 'srcc': 0.6698348129565218, 'srcc_status': 'DEFINED', 'lcc': 0.6851165274490757, 'lcc_status': 'DEFINED', 'ktau': 0.48663045904540275, 'ktau_status': 'DEFINED', 'mse': 3.159857019666854, 'mae': 1.441421172062556}, best_srcc=0.6698348129565218
Epoch 2: step=1874, train_loss=0.132400, validation={'score_scale': 'raw_0_to_10', 'sample_count': 3000, 'srcc': 0.6701622910604335, 'srcc_status': 'DEFINED', 'lcc': 0.6848812478715083, 'lcc_status': 'DEFINED', 'ktau': 0.4870614698526554, 'ktau_status': 'DEFINED', 'mse': 3.1653136705463183, 'mae': 1.4413591459592183}, best_srcc=0.6701622910604335
Epoch 3: step=2811, train_loss=0.130104, validation={'score_scale': 'raw_0_to_10', 'sample_count': 3000, 'srcc': 0.6710613225331425, 'srcc_status': 'DEFINED', 'lcc': 0.6868550977646717, 'lcc_status': 'DEFINED', 'ktau': 0.48805291446417454, 'ktau_status': 'DEFINED', 'mse': 3.137104932754793, 'mae': 1.435

{
  "status": "PASS",
  "device": "cuda:0",
  "epochs_completed": 50,
  "global_step": 46850,
  "best_srcc_raw_scale": 0.6736054520366399,
  "best_checkpoint": "/kaggle/working/moescore/runs/best_gate.pt",
  "latest_checkpoint": "/kaggle/working/moescore/runs/latest_gate.pt",
  "history": [
    {
      "epoch": 0,
      "global_step": 937,
      "train_loss": 0.13702678286024195,
      "validation_loss_normalized": 0.1830685337384542,
      "validation_metrics_raw": {
        "score_scale": "raw_0_to_10",
        "sample_count": 3000,
        "srcc": 0.6698348129565218,
        "srcc_status": "DEFINED",
        "lcc": 0.6851165274490757,
        "lcc_status": "DEFINED",
        "ktau": 0.48663045904540275,
        "ktau_status": "DEFINED",
        "mse": 3.159857019666854,
        "mae": 1.441421172062556
      },
      "learning_rate": 1e-05
    },
    {
      "epoch": 1,
      "global_step": 1874,
      "train_loss": 0.132399950083047,
      "validation_loss_normalized": 0.1835209581

## 14. Đánh giá validation

Chỉ dùng best checkpoint do validation SRCC chọn. Không dùng test labels.


In [21]:
if not SMOKE_ONLY and RUN_EVALUATION:
    best = RUNS_ROOT/"best_gate.pt"
    if not best.is_file(): raise FileNotFoundError("best_gate.pt chưa có; kiểm tra SRCC validation/training")
    run_cli(PROJECT_ROOT/"scripts/evaluate_moe_gating.py", "--config", gate_config,
            "--checkpoint", best, "--device", GATE_DEVICE,
            "--report-json", RUNS_ROOT/"validation_metrics.json")
    from data.moe_training_dataset import MoeTrainingDataset, collate_moe_features
    from data.moe_feature_store import preprocessing_contract
    from model.moe_gating_core import GatingNetwork
    from training.checkpoint_manager import MoeCheckpointManager
    from training.moe_evaluator import MoeGateEvaluator, write_submission_csv
    from torch.utils.data import DataLoader
    validation = MoeTrainingDataset(str(full_store), split="validation", require_labels=True)
    gate = GatingNetwork(input_feature_dim=validation.manifest["feature_width"], num_experts=9, profile="full_9").to(GATE_DEVICE)
    payload = MoeCheckpointManager().load_for_inference(best, gating_network=gate,
        expected_expert_checkpoint_provenance=validation.manifest["checkpoint_provenance"])
    if payload["feature_store_identity"]["validation"] != validation.identity: raise ValueError("Validation identity mismatch")
    if payload["feature_store_identity"]["preprocessing_contract"] != preprocessing_contract(validation.manifest): raise ValueError("Preprocessing identity mismatch")
    batches = DataLoader(validation, batch_size=config["validation"]["batch_size"], shuffle=False,
                         collate_fn=collate_moe_features)
    results = MoeGateEvaluator(gate).evaluate(batches)
    rows = [{"sample_id":sid,"wave_file_name":wav,"pred_score":float(score)} for sid,wav,score in
            zip(results["sample_ids"], results["wave_file_names"], results["prediction_raw"].tolist())]
    write_submission_csv(rows, RUNS_ROOT/"validation_predictions.csv")
    print(json.loads((RUNS_ROOT/"validation_metrics.json").read_text()))

RUN /usr/bin/python3 /kaggle/working/MOESCORE/scripts/evaluate_moe_gating.py --config /kaggle/working/moescore/configs/moe_gating_config.yaml --checkpoint /kaggle/working/moescore/runs/best_gate.pt --device cuda:0 --report-json /kaggle/working/moescore/runs/validation_metrics.json
{
  "status": "PASS",
  "device": "cuda:0",
  "checkpoint": "/kaggle/working/moescore/runs/best_gate.pt",
  "checkpoint_epoch": 26,
  "sample_count": 3000,
  "metrics_raw_scale": {
    "score_scale": "raw_0_to_10",
    "sample_count": 3000,
    "srcc": 0.6736054520366399,
    "srcc_status": "DEFINED",
    "lcc": 0.687931678184295,
    "lcc_status": "DEFINED",
    "ktau": 0.49036431869345165,
    "ktau_status": "DEFINED",
    "mse": 3.156416706975922,
    "mae": 1.4367924128373464
  },
  "regression_loss_normalized": 0.1827187068015337,
  "optimizer_steps": 0
}
{'status': 'PASS', 'device': 'cuda:0', 'checkpoint': '/kaggle/working/moescore/runs/best_gate.pt', 'checkpoint_epoch': 26, 'sample_count': 3000, 'metri

## 15. Dự đoán test không nhãn

Tạo `submission.csv` theo thứ tự gốc, không đọc nhãn test. Lưu thêm `test_predictions.csv`.


In [22]:
if not SMOKE_ONLY and RUN_TEST_INFERENCE:
    best = RUNS_ROOT/"best_gate.pt"
    if not best.is_file(): raise FileNotFoundError(best)
    run_cli(PROJECT_ROOT/"scripts/infer_moe_gating.py", "--config", gate_config,
            "--checkpoint", best, "--device", GATE_DEVICE,
            "--output", RUNS_ROOT/"submission.csv", "--report-json", RUNS_ROOT/"test_inference.json")
    shutil.copy2(RUNS_ROOT/"submission.csv", RUNS_ROOT/"test_predictions.csv")
    print("Submission:", RUNS_ROOT/"submission.csv")

RUN /usr/bin/python3 /kaggle/working/MOESCORE/scripts/infer_moe_gating.py --config /kaggle/working/moescore/configs/moe_gating_config.yaml --checkpoint /kaggle/working/moescore/runs/best_gate.pt --device cuda:0 --output /kaggle/working/moescore/runs/submission.csv --report-json /kaggle/working/moescore/runs/test_inference.json
{
  "status": "PASS",
  "device": "cuda:0",
  "sample_count": 3000,
  "checkpoint_epoch": 26,
  "output_csv": "/kaggle/working/moescore/runs/submission.csv",
  "labels_required": false
}
Submission: /kaggle/working/moescore/runs/submission.csv


# STAGE 15A — EVALUATE TEST SET AND SAVE JSON

In [23]:
import json
from pathlib import Path

if not SMOKE_ONLY and RUN_TEST_EVALUATION:

    print("=" * 65)
    print("MOESCORE — FINAL TEST EVALUATION")
    print("=" * 65)

    # 1. Best checkpoint selected using validation SRCC
    best_checkpoint = RUNS_ROOT / "best_gate.pt"

    if not best_checkpoint.is_file():
        raise FileNotFoundError(
            f"Best checkpoint not found: {best_checkpoint}"
        )

    # 2. Ground truth for test evaluation
    test_labels = DATASET_ROOT / "meta_data" / "test_average.csv"

    if not test_labels.is_file():
        raise FileNotFoundError(
            f"Test ground truth not found: {test_labels}"
        )

    # 3. Use the existing MOESCORE evaluation pipeline.
    #    This runs validation and label-free test inference,
    #    then matches test predictions with ground truth.
    evaluation_report = pipeline.evaluate_best(
        config=config,
        dataset_root=DATASET_ROOT,
        checkpoint=best_checkpoint,
    )

    # 4. JSON files are generated by evaluate_best()
    output_dir = Path(config["output_dir"])

    test_json = output_dir / "test_metrics.json"
    combined_json = output_dir / "metrics.json"

    if not test_json.is_file():
        raise FileNotFoundError(test_json)

    if not combined_json.is_file():
        raise FileNotFoundError(combined_json)

    # 5. Read saved results
    with test_json.open("r", encoding="utf-8") as f:
        test_result = json.load(f)

    metrics = test_result["metrics_raw_scale"]

    # 6. Print metrics
    print("\nTEST EVALUATION RESULTS")
    print("-" * 45)

    print("Samples :", metrics["sample_count"])
    print("SRCC    :", metrics["srcc"])
    print("LCC     :", metrics["lcc"])
    print("KTAU    :", metrics["ktau"])
    print("MSE     :", metrics["mse"])
    print("MAE     :", metrics["mae"])

    print("\nCheckpoint:", best_checkpoint)
    print("Test JSON :", test_json)
    print("Full JSON :", combined_json)

    print("\nPASS: Test evaluation completed!")

else:
    print("SKIPPED: Test evaluation disabled or smoke-only mode")

MOESCORE — FINAL TEST EVALUATION

TEST EVALUATION RESULTS
---------------------------------------------
Samples : 3000
SRCC    : 0.6533204214827315
LCC     : 0.7055141073113964
KTAU    : 0.4718471259490953
MSE     : 2.8031332287305055
MAE     : 1.3554817504882812

Checkpoint: /kaggle/working/moescore/runs/best_gate.pt
Test JSON : /kaggle/working/moescore/runs/test_metrics.json
Full JSON : /kaggle/working/moescore/runs/metrics.json

PASS: Test evaluation completed!


## 16. Artifacts và đường dẫn

Đối chiếu log, checkpoint, dự đoán và dataset URL. Smoke và full cache được lưu riêng.


In [24]:
artifacts_to_show = [LOG_ROOT/"smoke_extraction.json", LOG_ROOT/"smoke_gate.json",
                     LOG_ROOT/"full_extraction.json", LOG_ROOT/"full_store_validation.json",
                     LOG_ROOT/"upload.json", RUNS_ROOT/"best_gate.pt",
                     RUNS_ROOT/"validation_metrics.json", RUNS_ROOT/"submission.csv", RUNS_ROOT / "test_metrics.json", 
                     RUNS_ROOT / "metrics.json"]
for path in artifacts_to_show:
    print("PRESENT" if path.exists() else "SKIPPED", path)
print("Feature dataset:", f"https://www.kaggle.com/datasets/{FEATURE_DATASET_HANDLE}" if (LOG_ROOT/"upload.json").is_file() else "chưa upload")

PRESENT /kaggle/working/moescore/logs/smoke_extraction.json
PRESENT /kaggle/working/moescore/logs/smoke_gate.json
PRESENT /kaggle/working/moescore/logs/full_extraction.json
PRESENT /kaggle/working/moescore/logs/full_store_validation.json
SKIPPED /kaggle/working/moescore/logs/upload.json
PRESENT /kaggle/working/moescore/runs/best_gate.pt
PRESENT /kaggle/working/moescore/runs/validation_metrics.json
PRESENT /kaggle/working/moescore/runs/submission.csv
PRESENT /kaggle/working/moescore/runs/test_metrics.json
PRESENT /kaggle/working/moescore/runs/metrics.json
Feature dataset: chưa upload
